# Notebook 00 — NeuMF Baseline (Classic Collaborative Filtering)

**Baseline Cổ điển** cho bài toán Gợi ý.
Neural Matrix Factorization (NeuMF) kết hợp Generalized Matrix Factorization (GMF) 
và Multi-Layer Perceptron (MLP) dựa trên **User ID** và **Item ID** (News ID).

**Mục đích của Notebook này:**
Phép thử này nhằm chứng minh điểm yếu chí mạng của các phương pháp CF cổ điển 
trong hệ thống báo chí: **Vấn đề Cold-start**. Tin tức thường có tuổi thọ rất ngắn,
tập test/dev liên tục xuất hiện tin mới. Do NeuMF chỉ dựa trên ID mà không hiểu nội dung (title),
nó sẽ thất bại thảm hại khi dự đoán tin tức mới. Từ đó làm nổi bật sự cần thiết của
Deep Content-based Recommendation (NRMS, NRAGLS).

In [1]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, '.')
sys.path.append('/kaggle/input/datasets/neitng/utils-for-dl-major-assignment')

from utils import (
    seed_everything, TRAIN_DIR, DEV_DIR, WORK_DIR, MODEL_DIR, SEED,
    load_news, load_behaviors, count_parameters
)

seed_everything(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')

DEVICE: cuda


In [2]:
EMB_DIM = 64
NEG_K = 4
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 5

In [3]:
print("Loading data...")
news_train = load_news(TRAIN_DIR)
news_dev   = load_news(DEV_DIR)
news_all   = pd.concat([news_train, news_dev]).drop_duplicates('news_id').reset_index(drop=True)
beh_train  = load_behaviors(TRAIN_DIR)
beh_dev    = load_behaviors(DEV_DIR)

Loading data...


In [4]:
# Để chạy NeuMF, ta cần map mỗi user_id và news_id sang một số nguyên
print("Building ID mappings...")
all_users = pd.concat([beh_train['user_id'], beh_dev['user_id']]).unique()
user2idx = {u: i+1 for i, u in enumerate(all_users)}
NUM_USERS = len(user2idx)

all_nids = news_all['news_id'].unique()
nid2idx = {n: i+1 for i, n in enumerate(all_nids)}
NUM_NEWS = len(nid2idx)

print(f"Total Users: {NUM_USERS}, Total News: {NUM_NEWS}")

Building ID mappings...
Total Users: 94057, Total News: 65238


In [5]:
class NeuMFDataset(Dataset):
    def __init__(self, behaviors, user2idx, nid2idx, neg_k=4, max_rows=None):
        if max_rows: behaviors = behaviors.sample(n=max_rows, random_state=SEED)
        self.samples = []
        for _, row in behaviors.iterrows():
            uid = user2idx.get(row['user_id'], 0)
            imps = row['impressions'].split()
            pos = [i.split('-')[0] for i in imps if i.endswith('-1')]
            neg = [i.split('-')[0] for i in imps if i.endswith('-0')]
            if not pos or not neg: continue
            
            # Create training pairs
            for p in pos:
                p_idx = nid2idx.get(p, 0)
                if len(neg) >= neg_k:
                    n_samp = np.random.choice(neg, neg_k, replace=False)
                else:
                    n_samp = np.random.choice(neg, neg_k, replace=True)
                n_idx = [nid2idx.get(n, 0) for n in n_samp]
                self.samples.append((uid, p_idx, n_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        uid, p, n = self.samples[i]
        return (torch.tensor(uid, dtype=torch.long),
                torch.tensor(p, dtype=torch.long),
                torch.tensor(n, dtype=torch.long))

In [6]:
class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64):
        super().__init__()
        # GMF embeddings
        self.u_emb_gmf = nn.Embedding(num_users + 1, emb_dim, padding_idx=0)
        self.i_emb_gmf = nn.Embedding(num_items + 1, emb_dim, padding_idx=0)
        
        # MLP embeddings
        self.u_emb_mlp = nn.Embedding(num_users + 1, emb_dim, padding_idx=0)
        self.i_emb_mlp = nn.Embedding(num_items + 1, emb_dim, padding_idx=0)
        
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim // 2),
            nn.ReLU()
        )
        
        self.predict_layer = nn.Linear(emb_dim + (emb_dim // 2), 1)

    def forward(self, u, i):
        # u: (B,), i: (B,)
        u_gmf = self.u_emb_gmf(u)
        i_gmf = self.i_emb_gmf(i)
        gmf_vec = u_gmf * i_gmf
        
        u_mlp = self.u_emb_mlp(u)
        i_mlp = self.i_emb_mlp(i)
        mlp_vec = self.mlp(torch.cat([u_mlp, i_mlp], dim=-1))
        
        pred = self.predict_layer(torch.cat([gmf_vec, mlp_vec], dim=-1)).squeeze(-1)
        return pred

    def score_batch(self, u, P, N):
        # u: (B,), P: (B,), N: (B, K)
        B, K = N.shape
        pos_score = self.forward(u, P).unsqueeze(1) # (B, 1)
        
        u_exp = u.unsqueeze(1).expand(-1, K).reshape(-1) # (B*K,)
        N_flat = N.reshape(-1) # (B*K,)
        neg_score = self.forward(u_exp, N_flat).reshape(B, K) # (B, K)
        
        return torch.cat([pos_score, neg_score], dim=1) # (B, K+1)

In [7]:
model = NeuMF(NUM_USERS, NUM_NEWS, emb_dim=EMB_DIM).to(DEVICE)
n_params = count_parameters(model)
print(f"NeuMF parameters: {n_params:,}")

print("Building training dataset...")
train_ds = NeuMFDataset(beh_train, user2idx, nid2idx, neg_k=NEG_K, max_rows=150000)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

epoch_times = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    t0 = time.time()
    for u, p, n in train_dl:
        u, p, n = u.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
        optimizer.zero_grad()
        logits = model.score_batch(u, p, n)
        target = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
        loss = loss_fn(logits, target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    dt = time.time() - t0
    epoch_times.append(dt)
    print(f"Epoch {epoch}/{EPOCHS} | loss={np.mean(losses):.5f} | time={dt:.1f}s")

NeuMF parameters: 20,400,449
Building training dataset...
Epoch 1/5 | loss=1.50082 | time=13.2s
Epoch 2/5 | loss=1.43103 | time=12.6s
Epoch 3/5 | loss=1.39474 | time=12.7s
Epoch 4/5 | loss=1.33963 | time=12.4s
Epoch 5/5 | loss=1.25150 | time=12.4s


In [8]:
print("\nEvaluating on dev set...")
from utils import compute_ranking_metrics

model.eval()
all_auc, all_mrr, all_ndcg5, all_ndcg10 = [], [], [], []

with torch.no_grad():
    for _, row in beh_dev.iterrows():
        uid = user2idx.get(row['user_id'], 0)
        u_tensor = torch.tensor([uid], dtype=torch.long, device=DEVICE)
        
        imps = row['impressions'].split()
        cand_nids = [imp.split('-')[0] for imp in imps]
        labels = [int(imp.split('-')[1]) for imp in imps]
        
        cand_idx = [nid2idx.get(n, 0) for n in cand_nids]
        c_tensor = torch.tensor(cand_idx, dtype=torch.long, device=DEVICE)
        
        u_exp = u_tensor.expand(len(cand_idx))
        scores = model(u_exp, c_tensor).cpu().numpy()
        
        m = compute_ranking_metrics(labels, scores)
        if m is not None:
            all_auc.append(m['AUC'])
            all_mrr.append(m['MRR'])
            all_ndcg5.append(m['nDCG@5'])
            all_ndcg10.append(m['nDCG@10'])

metrics = {
    'AUC': np.mean(all_auc),
    'MRR': np.mean(all_mrr),
    'nDCG@5': np.mean(all_ndcg5),
    'nDCG@10': np.mean(all_ndcg10)
}

print("=" * 50)
print("NeuMF Baseline Results:")
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")
print("=" * 50)


Evaluating on dev set...
NeuMF Baseline Results:
  AUC: 0.549464
  MRR: 0.276913
  nDCG@5: 0.255933
  nDCG@10: 0.318103


In [9]:
results = {
    'model': 'NeuMF',
    'type': 'baseline',
    'complexity': 'O(1) inference, but O(U+I) memory',
    'parameters': n_params,
    'metrics': metrics,
    'epoch_times_sec': epoch_times,
    'avg_epoch_time_sec': float(np.mean(epoch_times)),
}

torch.save(model.state_dict(), MODEL_DIR / 'neumf_baseline.pt')
with open(WORK_DIR / 'neumf_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nModel saved to {MODEL_DIR / 'neumf_baseline.pt'}")
print(f"Results saved to {WORK_DIR / 'neumf_results.json'}")


Model saved to /kaggle/working/dl_results/models/neumf_baseline.pt
Results saved to /kaggle/working/dl_results/neumf_results.json
